### Framework


Requirements:

In [ ]:
%pip install \
hdbscan \
tensorflow \
tensorflow_hub \
scikit-learn \
pandas \
xlrd \
tqdm \
google_play_scraper \
sentence_transformers==3.1.1 \
transformers==4.38.2 \
torch \
torchvision \
umap-learn \
sastrawi \
nltk \
stanza \
gensim \
pyLDAvis \
XlsxWriter \
matplotlib \
plotly \
openpyxl \
setuptools \
seaborn

In [ ]:
import os
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
import pandas as pd
from ITER_DBSCAN import ITER_DBSCAN
from evaluation import EvaluateDataset
import re
import numpy as np
from sklearn.manifold import TSNE
import umap
from sklearn.cluster import AgglomerativeClustering
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import string
from collections import Counter
from nltk.corpus import stopwords
from nltk import download
from sklearn.feature_extraction.text import CountVectorizer
import stanza
import nltk
from nltk.corpus import stopwords
from nltk import download
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import pyLDAvis.gensim_models
import pyLDAvis
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import skew
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import StandardScaler
import json
import time
import pandas as pd
import seaborn as sns
import hdbscan
from sklearn.datasets import make_blobs

Stage-01: Load dataset

In [ ]:
filepath = "dataset.xlsx"
df = pd.read_excel(filepath)

Stage-02: Create Clean Dataset

In [ ]:
filepath = "dataset_anonim_phase7_2024_anonim_NER_openAI_2.xlsx"
df = pd.read_excel(filepath)

download("stopwords")
STOPWORDS_ID = set(stopwords.words("indonesian"))
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# preprocessing with lemmatization
def clean_text_deep(text):

    if not isinstance(text, str):
        return text

    # 1. lowercase
    text = text.lower()

    # 2. replace year 1900-2050
    text = re.sub(r'\b(19\d{2}|20[0-4]\d|2050)\b', ' tahun ', text)

    # 3. replace numbers
    text = re.sub(r'\b\d+\b', ' ', text)

    # 4. replace punctuation
    text = re.sub(r'[^\w\s<>]', ' ', text)

    # 5. remove ALL non-letter characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 6. remove single-character tokens
    text = re.sub(r'\b[a-z]\b', ' ', text)

    # 7. clean double spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # 8. Remove stopwords
    tokens = text.split()
    tokens = [
        word for word in tokens
        if word not in STOPWORDS_ID
    ]

    # 9. Lemmatization
        
    # include stopword
    # text = " ".join([stemmer.stem(word) for word in text.split()])
    
    # do not include stopword
    # tokens = [stemmer.stem(word) for word in tokens] # uncomment if use lemmatization
    text = " ".join(tokens)

    return text

def clean_text_simple(text):

    if not isinstance(text, str):
        return text

    # 1. lowercase
    text = text.lower()

    # 2. replace year 1900-2050
    text = re.sub(r'\b(19\d{2}|20[0-4]\d|2050)\b', ' ', text)

    # 3. replace numbers
    text = re.sub(r'\b\d+\b', ' ', text)

    # 4. replace punctuation
    text = re.sub(r'[^\w\s<>]', ' ', text)

    # 5. remove ALL non-letter characters (termasuk > < / dll)
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 6. remove single-character tokens (s, d, x, dll)
    text = re.sub(r'\b[a-z]\b', ' ', text)

    # 7. clean double spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# clean all data
# for col in df.select_dtypes(include=['object']).columns:
#    df[col] = df[col].apply(clean_text)

# clean certain column data
df['question_clean_simple'] = df['question'].apply(clean_text_simple)

df['question_clean_deep'] = df['question'].apply(clean_text_deep)
# preview
print(df[['question', 'question_clean_simple', 'question_clean_deep']].head(10))

# save to excel for later use
output_file = "dataset_anonim_phase7_clean.xlsx"
df.to_excel(output_file, index=False)
print(f"\nFile save to: {output_file}")


Stage-03: Preview Dataset

In [ ]:
print('Before: ', len(df))
df = df.dropna()
print('After: ', len(df))
df = df.reset_index()
del df['index']
df.head(5)

Stage-04: extract entity words from ner column (create entities column)

In [ ]:
def extract_entities_from_ner(text):
    if pd.isna(text):
        return None
    
    parts = text.split(":", 1)
    if len(parts) < 2:
        return None
    
    sentence_part = parts[1]
    entities = re.findall(r'\[(.*?)\]', sentence_part)
    
    return " ".join(entities)

# Apply to dataframe
df['entities'] = df['ner'].apply(extract_entities_from_ner)

# Preview
print(df[['ner', 'entities']].head())

Stage-05: identify K topic for LDA NER topic

In [ ]:

# 1. PREPROCESS NER TEXT
token_ner_list = df['entities'].apply(lambda x: x.split()).tolist()

# 2. BUILD DICTIONARY & CORPUS
dict_ner = corpora.Dictionary(token_ner_list)
bow_corpus_ner = [dict_ner.doc2bow(doc) for doc in token_ner_list]

# 3. FIND BEST NUMBER OF TOPICS (COHERENCE)
coherence_result = {}

for num_topic_try in range(2, 31):
    lda_temp_model = LdaModel(
        corpus=bow_corpus_ner,
        id2word=dict_ner,
        num_topics=num_topic_try,
        random_state=42,
        passes=10
    )
    
    coherence_model_tmp = CoherenceModel(
        model=lda_temp_model,
        texts=token_ner_list,
        dictionary=dict_ner,
        coherence='c_v'
    )
    
    coh_score = coherence_model_tmp.get_coherence()
    coherence_result[num_topic_try] = coh_score
    
    print(f"K={num_topic_try}, Coherence={coh_score:.4f}")

Stage-06: Built vector LDA topic from NER entities

In [ ]:
# 4. TRAIN FINAL LDA MODEL
BEST_K = 3  

lda_final_model = LdaModel(
    corpus=bow_corpus_ner,
    id2word=dict_ner,
    num_topics=BEST_K,
    random_state=42,
    passes=10
)

# 5. BUILD TOPIC VECTOR (DOC-TOPIC DISTRIBUTION)
def generate_topic_vector(model, corpus_data):
    topic_vectors = []
    
    for bow_doc in corpus_data:
        topic_distribution = model.get_document_topics(
            bow_doc, 
            minimum_probability=0
        )
        
        vec = np.array(
            [prob for _, prob in sorted(topic_distribution, key=lambda x: x[0])],
            dtype=np.float32
        )
        
        topic_vectors.append(vec)
    
    return topic_vectors

vectors_topic = generate_topic_vector(lda_final_model, bow_corpus_ner)

print(f"Total Data : {len(vectors_topic)}")
print(f"Dim Vector : {vectors_topic[0].shape[0]}")
print(f"Sample Vec :\n{vectors_topic[0]}")

# 6. EXTRACT TOP WORDS PER TOPIC
topic_word_list = []

for topic_id in range(BEST_K):
    terms = lda_final_model.show_topic(topic_id, topn=50)
    topic_word_list.append([word for word, _ in terms])

df_topic_summary = pd.DataFrame({
    "Topic_ID": [f"Topic_{i}" for i in range(BEST_K)],
    "Top_Entities": [", ".join(words) for words in topic_word_list]
})

print("\nPreview Top Entities per Topic:")
print(df_topic_summary)

file_output_topic = "LDA_NER_Topics.xlsx"
df_topic_summary.to_excel(file_output_topic, index=False)
print(f"\nSaved to: {file_output_topic}")

# 7. ASSIGN TOPIC
def assign_dominant_topic(model, corpus_data):
    topic_labels = []
    
    for bow_doc in corpus_data:
        topic_distribution = model.get_document_topics(bow_doc)
        
        # select topic with highest prob
        dominant_topic = max(topic_distribution, key=lambda x: x[1])[0]
        topic_labels.append(dominant_topic)
    
    return topic_labels

df['topic_lda'] = assign_dominant_topic(lda_final_model, bow_corpus_ner)

output_full = "LDA_NER_with_topic.xlsx"
df.to_excel(output_full, index=False)

print(f"\nFinal dataset saved to: {output_full}")

Stage-07: Create vector ner

In [ ]:
# STEP 1: BUILD NER VOCAB (FROM df.entities)

entity_texts = df["entities"].astype(str).tolist()

all_tags = set()

for text in entity_texts:

    tokens = text.split()
    clean_tokens = [t.strip() for t in tokens if t.strip() != ""]
    all_tags.update(clean_tokens)

ner_vocab = sorted(list(all_tags))

print("NER Vocab:")
print(ner_vocab)
print(f"Total unique tags: {len(ner_vocab)}")

# STEP 2: BUILD INDEX MAPPING
tag_to_index = {tag: i for i, tag in enumerate(ner_vocab)}

# STEP 3: BUILD VECTORS (FROM df.entities)
vectors_ner = []

entity_texts = df["entities"].fillna("").astype(str).tolist()

for text in entity_texts:
    # init vector nol
    vec = np.zeros(len(ner_vocab), dtype=np.float32)
    tokens = text.split()

    # freq
    for token in tokens:
        token_clean = token.strip()
        if token_clean in tag_to_index:
            idx = tag_to_index[token_clean]
            vec[idx] += 1

    vectors_ner.append(vec)

# convert ke numpy array 2D
vectors_ner = np.array(vectors_ner, dtype=np.float32)

print("\nShape vectors_ner:", vectors_ner.shape)
print("Preview vectors_ner[0]:")
print(vectors_ner[0])

Stage-08: Scaling vector ner

In [ ]:
# scaling vectors_ner

vectors_ner_array = np.vstack([np.ravel(v) for v in vectors_ner])  # (n_samples, n_features)

scaler = PowerTransformer()
vectors_ner_power_array = scaler.fit_transform(vectors_ner_array)
vectors_ner_power = [v for v in vectors_ner_power_array]

print("Shape:", vectors_ner_power_array.shape)
print("Preview first vector:", vectors_ner_power[0])

Stage-09: Build semantic vector from OpenAI Embedding (load from JSON)

In [ ]:
# load OpenAI Embeddings Result (JSON format file)

def load_embeddings_with_validation(json_file, df):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    texts = []
    vectors = []

    # ensure same in record length
    if len(data) != len(df):
        print(f"[WARNING] JSON length ({len(data)}) != DataFrame ({len(df)})")

    # loop paralel
    for i in range(min(len(data), len(df))):
        json_item = data[i]
        df_row = df.iloc[i]

        json_id = json_item.get("id")
        json_text = str(json_item.get("text"))
        json_emb = json_item.get("embedding")

        df_id = df_row["id"]
        df_text = str(df_row["question_clean_simple"])

        # validate
        if json_id == df_id and json_text == df_text:
            texts.append(json_text)
            vectors.append(json_emb)
        else:
            print(f"[MISMATCH] Row: {i}")
            print(f"JSON -> id: {json_id}, text: {json_text}")
            print(f"DF   -> id: {df_id}, text: {df_text}\n")

    vectors = np.array(vectors, dtype=np.float32)

    return texts, vectors

if __name__ == "__main__":

    # ensure id column
    if "id" not in df.columns:
        df["id"] = df.index

    texts, vectors_vw = load_embeddings_with_validation(
        "OpenAIEmbeddings_512_question_clean_simple.json",
        df
    )

    print("Shape:", vectors_vw.shape)
    print("First vector:", vectors_vw[0])

Stage-10: Build Semantic Vector with IndoSBERT

In [ ]:
# Build Semantic Vector with IndoSBERT

# Select Column
dataset_1A = df.question.values.tolist() 

from IndoSBERTEmbedding import IndoSBERTEmbedding

embedding = IndoSBERTEmbedding()
vectors_vw = embedding.getEmbeddings(dataset_1A)

print("Item(s):", len(vectors_vw))
print("Length :", len(vectors_vw[0]))
print("Preview vectors_vw:")
vectors_vw[0]

Stage-11: Build Semantic Vector with MiniLMEmbedding

In [ ]:
# Build Semantic Vector with MiniLMEmbedding

# Select Column
dataset_1B = df.question_clean.values.tolist()

from MiniLMEmbedding import MiniLMEmbedding

embedding = MiniLMEmbedding()
vectors_vw = embedding.getEmbeddings(dataset_1B)

print("Item(s):", len(vectors_vw))
print("Length :", len(vectors_vw[0]))
print("Preview vectors_vw:")
vectors_vw[0]

Stage-12: Build Semantic Vector with SentenceEmbedding (deprecated)

In [ ]:
# Build Semantic Vector with SentenceEmbedding

# Select Column: question, question_clean
# dataset_1C = df.question_clean.values.tolist()

# from sentenceEmbedding import SentenceEmbedding

# embedding = SentenceEmbedding()
# vectors_vw = embedding.getEmbeddings(dataset_1C)

# print("Item(s):", len(vectors_vw))
# print("Length :", len(vectors_vw[0]))
# print("Preview vectors_vw:")
# vectors_vw[0]

Stage-13: Normalize vectors_vw

In [ ]:
# Normalize vectors_vw
vectors_vw_normal = []

for v in vectors_vw:
    v_np = np.array(v, dtype=np.float32)
    
    # norm L2
    norm_rec = np.linalg.norm(v_np)
    
    # error handler divide by 0
    if norm_rec == 0:
        norm_vec = v_np
    else:
        norm_vec = v_np / norm_rec
    
    vectors_vw_normal.append(norm_vec)

print("Shape:", np.array(vectors_vw_normal).shape)

# =========================
# 1. CEK NORM (PANJANG VEKTOR)
# =========================
norms = np.linalg.norm(vectors_vw_normal, axis=1)

print("\n=== VECTOR NORM ===")
print("Min norm :", norms.min())
print("Max norm :", norms.max())
print("Mean norm:", norms.mean())
print("Std norm :", norms.std())

print("Sample norms:", norms[:10])

# plotting aman (anti error)
if np.std(norms) < 1e-6:
    print("Data is L2-normalized")

    plt.plot(norms[:100])
    plt.title("Norm Values (Flat / Identical)")
    plt.ylabel("Norm")
    plt.xlabel("Sample Index")
    plt.show()
else:
    bins = min(30, max(5, int(len(norms)/10)))
    plt.hist(norms, bins=bins)
    plt.title("Distribution of Vector Norms")
    plt.xlabel("Norm")
    plt.ylabel("Frequency")
    plt.show()

Stage-14: Build top_ngram_features

In [ ]:
# build top_ngram_features

# Select Column: question, question_clean
dataset_2 = df.question_clean_deep.values.tolist() #sentences with deep clean, remove stopwords

#configuration
TOP_K = 30
NGRAM_RANGE = [1, 2, 3, 4]

# build global top-K ngram
top_ngram_features = {}
top_ngram_tables = []

for n in NGRAM_RANGE:
    vectorizer = CountVectorizer(
        ngram_range=(n, n),
        token_pattern=r"(?u)\b\w+\b"
    )

    X = vectorizer.fit_transform(dataset_2)
    vocab = vectorizer.get_feature_names_out()
    counts = X.sum(axis=0).A1

    freq = dict(zip(vocab, counts))
    top_items = Counter(freq).most_common(TOP_K)

    # save top-K ngram
    top_ngram_features[n] = [k for k, _ in top_items]

    for rank, (ngram, count) in enumerate(top_items, start=1):
        top_ngram_tables.append({
            "ngram_size": n,
            "rank": rank,
            "ngram": ngram,
            "frequency": int(count)
        })

df_top_ngram = pd.DataFrame(top_ngram_tables)
df_top_ngram.to_excel("top_ngram_features.xlsx", index=False)
print("Top-K ngram features saved to: top_ngram_features.xlsx")
print(df_top_ngram.head(10))

Stage-15

In [ ]:
# LOAD N-GRAM REFERENCE
dataset_2 = df.question_clean_deep.values.tolist()
df_ref = pd.read_excel("top_ngram_features.xlsx")

#configuration
TOP_K = 30
NGRAM_RANGE = [1, 2, 3, 4]

top_ngram_features = {}
ngram_dimensions = {}

for n in NGRAM_RANGE:
    features = (
        df_ref[df_ref["ngram_size"] == n]
        .sort_values("rank")
        ["ngram"]
        .tolist()
    )
    top_ngram_features[n] = features
    ngram_dimensions[n] = len(features)

# CHECK LOADED FEATURES
print("\nCheck loaded n-gram features:")

for n in NGRAM_RANGE:
    print(f"\n{n}-gram features loaded: {ngram_dimensions[n]}")

    if ngram_dimensions[n] > 0:
        print("Example features:")
        print(top_ngram_features[n][:TOP_K])
    else:
        print("WARNING: No features found for this n-gram size")


total_features = sum(ngram_dimensions.values())

print("\nSummary:")
print("Total ngram features:", total_features)
print("Feature distribution per n:", ngram_dimensions)

Stage-16

In [ ]:
# BUILD LEXICAL VECTOR PER ROW

def build_vk(text: str):

    vk_final = []

    for n in NGRAM_RANGE:

        vectorizer = CountVectorizer(
            ngram_range=(n, n),
            vocabulary=top_ngram_features[n],
            token_pattern=r"(?u)\b\w+\b"
        )

        X = vectorizer.fit_transform([text])

        vec = X.toarray()[0]   # numpy array
        vk_final.extend(vec)

    return np.array(vk_final, dtype=np.float32)

# BUILD VECTOR FOR ALL ROWS
vectors_vk = []

for text in dataset_2:
    v = build_vk(text)
    vectors_vk.append(v)

# CHECK RESULT
print("items :", len(vectors_vk))
print("length:", len(vectors_vk[0]))
print("Preview vectors_vk:")
vectors_vk

Stage-17

In [ ]:
# Scale vk with PowerTransformer

# list of arrays 2D array
vectors_vk_array = np.vstack([np.ravel(v) for v in vectors_vk])  # shape (n_samples, n_features)
scaler = PowerTransformer()

# Fit dan transform
vectors_vk_power_array = scaler.fit_transform(vectors_vk_array)  # hasil 2D (n_samples, n_features)

vectors_vk_power = [v for v in vectors_vk_power_array]

print("Shape:", vectors_vk_power_array.shape)
print("Preview first vector:", vectors_vk_power[0])

Stage-18: build a syntactic vector (POSTags) per row using Stanza

In [ ]:
# build a syntactic vector (POSTags) per row using Stanza

# Select Column: 
dataset_3 = df["question_clean_simple"].tolist()

dataset_3[0:5]

# Define ALL UPOS tags based on Universal POS Tagset
ALL_UPOS = [
    "PRON",   # pronoun (saya, dia)
    "PROPN",  # proper noun (BPS, Jakarta)
    "NOUN",   # common noun (data, desa)
    "NUM",    # number (2020, 100)
    "ADJ",    # adjective (baru, tinggi)
    "DET",    # determiner (setiap, semua)
    "VERB",   # main verb (mengirim, menanyakan)
    "AUX",    # auxiliary verb (sedang, akan)
    "ADV",    # adverb (cepat, tidak)
    "ADP",    # adposition / preposition (di, dari, ke)
    "CCONJ",  # coordinating conjunction (dan, atau)
    "SCONJ",  # subordinating conjunction (karena, ketika)
    "PART",   # particle (lah, pun)
    "INTJ",   # interjection (hai, oh)
    "PUNCT",  # punctuation (., ?, !)
    "SYM",    # symbol ($, %, +)
    "X"       # other / unknown
]

# SETUP STANZA (INDONESIAN)
stanza.download("id") 
nlp = stanza.Pipeline(
    lang="id",
    processors="tokenize,pos",
    use_gpu=True,
    verbose=False
)

def build_vp_vector(text: str):
    doc = nlp(text)
    pos_counts = {p: 0 for p in ALL_UPOS}
    pos_detail = []

    for sent in doc.sentences:
        for word in sent.words:
            upos = word.upos
            pos_detail.append(f"{word.text}/{upos}")
            if upos in pos_counts:
                pos_counts[upos] += 1

    vp_vector = np.array([pos_counts[p] for p in ALL_UPOS], dtype=np.float32)
    return vp_vector, " ".join(pos_detail)

# APPLY TO DATASET
vectors_vp = []
pos_details = []

for text in dataset_3:
    vp_vec, detail = build_vp_vector(text)
    vectors_vp.append(vp_vec)
    pos_details.append(detail)

# SUMMARY
print(f"Length   : {len(vectors_vp)}")               
print(f"Dimension: {vectors_vp[0].shape[0]}")       
print(f"Preview vector (first): {vectors_vp[0]}")
print(f"POS detail (first 10 tokens of first sentence): {pos_details[0].split()[:10]}")

# SAVE POS DETAIL TO EXCEL
df_pos = pd.DataFrame({
    "question_clean_simple": dataset_3,
    "pos_detail": pos_details
})

df_pos.to_excel("pos_detail.xlsx", index=False)
print("\nPOS detail saved to pos_detail.xlsx")

Stage-19

In [ ]:
# Scale vp with PowerTransformer

# list of arrays 2D array
vectors_vp_array = np.vstack([np.ravel(v) for v in vectors_vp])  # shape (n_samples, n_features)
scaler = PowerTransformer()

# Fit dan transform
vectors_vp_power_array = scaler.fit_transform(vectors_vp_array)  # hasil 2D (n_samples, n_features)
vectors_vp_power = [v for v in vectors_vp_power_array]

print("Shape:", vectors_vp_power_array.shape)
print("Preview first vector:", vectors_vp_power[0])


Stage-20

In [ ]:
# build topic vector per row using LDA: Find the number of topics (K) with the highest coherence score

dataset_4 = df.question_clean_deep.values.tolist()
dataset_tokens = [text.split() for text in dataset_4] 

# BUILD DICTIONARY & CORPUS FOR LDA
dictionary = corpora.Dictionary(dataset_tokens)
corpus = [dictionary.doc2bow(text) for text in dataset_tokens]

# CALCULATE COHERENCE for K
coherence_scores = {}
for k_test in range(2, 11):
    lda_model_tmp = LdaModel(corpus=corpus,
                             id2word=dictionary,
                             num_topics=k_test,
                             random_state=42,
                             passes=10)
    coherencemodel = CoherenceModel(model=lda_model_tmp,
                                    texts=dataset_tokens,
                                    dictionary=dictionary,
                                    coherence='c_v')
    score = coherencemodel.get_coherence()
    coherence_scores[k_test] = score
    print(f"K={k_test}, Coherence={score:.4f}")

Stage-21

In [ ]:
dataset_4 = df.question_clean_deep.values.tolist()
dataset_tokens = [text.split() for text in dataset_4] 

# BUILD DICTIONARY & CORPUS FOR LDA
dictionary = corpora.Dictionary(dataset_tokens)
corpus = [dictionary.doc2bow(text) for text in dataset_tokens]

K = 8 
lda_model = LdaModel(corpus=corpus,
                     id2word=dictionary,
                     num_topics=K,
                     random_state=42,
                     passes=10)

def build_vt_vector(lda_model, corpus):
    vt_vectors = []
    for doc_bow in corpus:
        topic_dist = lda_model.get_document_topics(doc_bow, minimum_probability=0)
        vec = np.array([prob for _, prob in sorted(topic_dist, key=lambda x: x[0])],
                       dtype=np.float32)
        vt_vectors.append(vec)
    return vt_vectors

vectors_vt = build_vt_vector(lda_model, corpus)

print(f"Length   : {len(vectors_vt)}")
print(f"Dimension: {vectors_vt[0].shape[0]}")
print(f"Preview  :\n{vectors_vt[0]}")

top_words_per_topic = []
for i in range(K):
    topic_terms = lda_model.show_topic(i, topn=50)  # top 50
    top_words_per_topic.append([word for word, _ in topic_terms])

df_topics = pd.DataFrame({
    "Topic": [f"Topic {i}" for i in range(K)],
    "Top_Words": [", ".join(words) for words in top_words_per_topic]
})

print("\nPreview top words per topic:")
print(df_topics)

output_file = "LDA_top_Words_Per_Topic.xlsx"
df_topics.to_excel(output_file, index=False)
print(f"\nFile saved to: {output_file}")

Stage-22

In [ ]:
# normalize vectors_vt
vectors_vt_normal = []

for v in vectors_vt:
    v_np2 = np.array(v, dtype=np.float32)
    
    # norm L2
    norm_rec2 = np.linalg.norm(v_np2)
    
    # error handler divide by 0
    if norm_rec2 == 0:
        norm_vec2 = v_np2
    else:
        norm_vec2 = v_np2 / norm_rec2
    
    vectors_vt_normal.append(norm_vec2)

print("Shape:", np.array(vectors_vt_normal).shape)

# 1. CHECK NORM (vector length)
norms = np.linalg.norm(vectors_vt_normal, axis=1)

print("\n=== VECTOR NORM ===")
print("Min norm :", norms.min())
print("Max norm :", norms.max())
print("Mean norm:", norms.mean())
print("Std norm :", norms.std())

print("Sample norms:", norms[:10])

#plotting
if np.std(norms) < 1e-6:
    print("Data is L2-normalized")

    plt.plot(norms[:100])
    plt.title("Norm Values (Flat / Identical)")
    plt.ylabel("Norm")
    plt.xlabel("Sample Index")
    plt.show()
else:
    bins = min(30, max(5, int(len(norms)/10)))
    plt.hist(norms, bins=bins)
    plt.title("Distribution of Vector Norms")
    plt.xlabel("Norm")
    plt.ylabel("Frequency")
    plt.show()

Stage-23

In [ ]:
# Concat vectors (vw, vk, vp, vt, vner)

def concat_vectors(list_vectors, weights=None):
    
    n = len(list_vectors[0])

    for v in list_vectors:
        if len(v) != n:
            raise ValueError("All vectors must have the same number of items.")

    # default weight = 1
    if weights is None:
        weights = [1.0] * len(list_vectors)

    # validasi weight
    if len(weights) != len(list_vectors):
        raise ValueError("Length of weights must match number of vector groups.")

    vectors_concat = []

    for i in range(n):

        parts = []
        for v, w in zip(list_vectors, weights):
            vec = np.array(v[i], dtype=np.float32)
            
            # apply weight
            vec_weighted = vec * w
            
            parts.append(vec_weighted)

        merged = np.concatenate(parts)
        vectors_concat.append(merged)

    return vectors_concat

weights = [1, 1, 1, 1]
selected_vectors = [vectors_vw, vectors_vk_power, vectors_vp_power, vectors_vt_normal]
vectors_concat = concat_vectors(selected_vectors, weights=weights)

print("Check weighted norms:")
for i, (v, w) in enumerate(zip(selected_vectors, weights), start=1):
    norm = np.linalg.norm(v[0])
    print(f"Vector {i} | weight={w} | norm(before)={norm:.4f} | norm(after)={norm*w:.4f}")

vectors_concat

Stage-24

In [ ]:
print("Shape:", np.array(vectors_concat).shape)

Stage-25

In [ ]:
# Global normalization

vectors_norm = []

for v in vectors_concat:
    v_np = np.array(v, dtype=np.float32)
    
    # norm L2
    norm = np.linalg.norm(v_np)
    
    if norm == 0:
        norm_vec = v_np
    else:
        norm_vec = v_np / norm
    
    vectors_norm.append(norm_vec)
    
print(f"Length   : {len(vectors_norm)}")
print(f"Dimension: {len(vectors_norm[0])}")

# vectors_norm = normalize(vectors_concat, norm='l2')
# print(f"Length   : {len(vectors_norm)}")
# print(f"Dimension: {len(vectors_norm[0])}")

vectors_norm[0]

norms = np.linalg.norm(vectors_norm, axis=1)

print("\n=== VECTOR NORM ===")
print("Min norm :", norms.min())
print("Max norm :", norms.max())
print("Mean norm:", norms.mean())
print("Std norm :", norms.std())

print("Sample norms:", norms[:10])

# plotting 
if np.std(norms) < 1e-6:
    print("Norm hampir identik → kemungkinan sudah L2-normalized")

    plt.plot(norms[:100])
    plt.title("Norm Values (Flat / Identical)")
    plt.ylabel("Norm")
    plt.xlabel("Sample Index")
    plt.show()
else:
    bins = min(30, max(5, int(len(norms)/10)))
    plt.hist(norms, bins=bins)
    plt.title("Distribution of Vector Norms")
    plt.xlabel("Norm")
    plt.ylabel("Frequency")
    plt.show()


# 2. VARIANCE PER DIMENSION
vw = np.array(vectors_norm)  # shape (n_samples, n_features)
variances = np.var(vw, axis=0)

print("\n=== VARIANCE PER DIMENSION ===")
print("Min variance :", variances.min())
print("Max variance :", variances.max())
print("Mean variance:", variances.mean())
print("Std variance :", variances.std())

plt.plot(variances)
plt.title("Variance per Dimension")
plt.xlabel("Dimension Index")
plt.ylabel("Variance")
plt.show()

# 3. CHECK VALUE RANGE
min_vals = vw.min(axis=0)
max_vals = vw.max(axis=0)

print("\n=== VALUE RANGE ===")
print("Global min:", min_vals.min())
print("Global max:", max_vals.max())


# 4. CHECK COSINE SIMILARITY
sample_size = min(300, len(vw)) 
vw_sample = vw[:sample_size]

cos_sim = cosine_similarity(vw_sample)
cos_sim_vals = cos_sim[np.triu_indices_from(cos_sim, k=1)]

print("\n=== COSINE SIMILARITY ===")
print("Min:", cos_sim_vals.min())
print("Max:", cos_sim_vals.max())
print("Mean:", cos_sim_vals.mean())
print("Std :", cos_sim_vals.std())

# plot
if np.std(cos_sim_vals) < 1e-6:
    print("Cosine similarity hampir identik")

    plt.plot(cos_sim_vals[:100])
    plt.title("Cosine Similarity (Flat)")
    plt.show()
else:
    bins = min(30, max(5, int(len(cos_sim_vals)/20)))
    plt.hist(cos_sim_vals, bins=bins)
    plt.title("Cosine Similarity Distribution")
    plt.xlabel("Similarity")
    plt.ylabel("Frequency")
    plt.show()

Stage-26

In [ ]:
# Scale the Data

scaler = StandardScaler()
vectors_norm_scaled = scaler.fit_transform(vectors_norm)
print(vectors_norm[0])
print(vectors_norm_scaled[0])

Stage-27: Dimensionality Reduction

In [ ]:
# Dimensionality Reduction

def reduce_dimensionality(vectors_source, method='umap', target_dim=120, random_state=42):
    
    X = np.stack([np.array(v, dtype=np.float32) for v in vectors_source], axis=0)
    print("Input shape:", X.shape)
    
    # method selection
    if method.lower() == 'umap':
        reducer = umap.UMAP(
            n_components=target_dim,
            random_state=random_state,
            n_neighbors=30,
            min_dist=0.0,
            metric='cosine' #'euclidean'
        )
    elif method.lower() == 'tsne':
        reducer = TSNE(
            n_components=target_dim,
            method='exact',  #barnes-hut
            random_state=random_state,
            init='pca'
        )
    else:
        raise ValueError("Method must be 'umap' or 'tsne'")
    
    # Fit & transform → output shape = (n_samples, target_dim)
    red_vectors_2d = reducer.fit_transform(X)
    
    # list of np.array
    red_vectors_list = [np.array(v, dtype=np.float32) for v in red_vectors_2d]
        
    return red_vectors_list

vectors_reduce = reduce_dimensionality(vectors_vw, method='umap', target_dim=120)

# output
print("output shape: (", len(vectors_reduce), ", ", len(vectors_reduce[0]), ")")
vectors_reduce[0]

Stage-28

In [ ]:
# Global normalization after reduce dimensionality

vectors_norm = []

for v in vectors_reduce:
    v_np = np.array(v, dtype=np.float32)
    
    # norm L2
    norm = np.linalg.norm(v_np)
    
    if norm == 0:
        norm_vec = v_np
    else:
        norm_vec = v_np / norm
    
    vectors_norm.append(norm_vec)
    
print(f"Length   : {len(vectors_norm)}")
print(f"Dimension: {len(vectors_norm[0])}")

# vectors_norm = normalize(vectors_concat, norm='l2')
# print(f"Length   : {len(vectors_norm)}")
# print(f"Dimension: {len(vectors_norm[0])}")

vectors_norm[0]

norms = np.linalg.norm(vectors_norm, axis=1)

print("\n=== VECTOR NORM ===")
print("Min norm :", norms.min())
print("Max norm :", norms.max())
print("Mean norm:", norms.mean())
print("Std norm :", norms.std())

print("Sample norms:", norms[:10])

# plotting
if np.std(norms) < 1e-6:
    print("Data is L2-normalized")

    plt.plot(norms[:100])
    plt.title("Norm Values (Flat / Identical)")
    plt.ylabel("Norm")
    plt.xlabel("Sample Index")
    plt.show()
else:
    bins = min(30, max(5, int(len(norms)/10)))
    plt.hist(norms, bins=bins)
    plt.title("Distribution of Vector Norms")
    plt.xlabel("Norm")
    plt.ylabel("Frequency")
    plt.show()


vw = np.array(vectors_norm)  # shape (n_samples, n_features)
variances = np.var(vw, axis=0)

print("\n=== VARIANCE PER DIMENSION ===")
print("Min variance :", variances.min())
print("Max variance :", variances.max())
print("Mean variance:", variances.mean())
print("Std variance :", variances.std())

plt.plot(variances)
plt.title("Variance per Dimension")
plt.xlabel("Dimension Index")
plt.ylabel("Variance")
plt.show()

min_vals = vw.min(axis=0)
max_vals = vw.max(axis=0)

print("\n=== VALUE RANGE ===")
print("Global min:", min_vals.min())
print("Global max:", max_vals.max())


sample_size = min(300, len(vw))
vw_sample = vw[:sample_size]

cos_sim = cosine_similarity(vw_sample)
cos_sim_vals = cos_sim[np.triu_indices_from(cos_sim, k=1)]

print("\n=== COSINE SIMILARITY ===")
print("Min:", cos_sim_vals.min())
print("Max:", cos_sim_vals.max())
print("Mean:", cos_sim_vals.mean())
print("Std :", cos_sim_vals.std())

# plot
if np.std(cos_sim_vals) < 1e-6:
    print("Cosine similarity hampir identik")

    plt.plot(cos_sim_vals[:100])
    plt.title("Cosine Similarity (Flat)")
    plt.show()
else:
    bins = min(30, max(5, int(len(cos_sim_vals)/20)))
    plt.hist(cos_sim_vals, bins=bins)
    plt.title("Cosine Similarity Distribution")
    plt.xlabel("Similarity")
    plt.ylabel("Frequency")
    plt.show()

Stage-29: Visualize data vectors with dimensionality reduction dim=3

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def reduce_dimensionality(vectors_source, method='umap', target_dim=3, random_state=42):
    
    X = np.stack([np.array(v, dtype=np.float32) for v in vectors_source], axis=0)
    print("Input shape:", X.shape)
    
    # method selection
    if method.lower() == 'umap':
        reducer = umap.UMAP(
            n_components=target_dim,
            random_state=random_state,
            n_neighbors=30,
            min_dist=0.0,
            metric='cosine' #'euclidean'
        )
    elif method.lower() == 'tsne':
        reducer = TSNE(
            n_components=target_dim,
            method='exact',  #barnes-hut
            random_state=random_state,
            init='pca'
        )
    else:
        raise ValueError("Method must be 'umap' or 'tsne'")
    
    # Fit & transform → output shape = (n_samples, target_dim)
    red_vectors_2d = reducer.fit_transform(X)
    
    # list of np.array
    red_vectors_list = [np.array(v, dtype=np.float32) for v in red_vectors_2d]
        
    return red_vectors_list

vectors_viz= reduce_dimensionality(vectors_norm, method='umap', target_dim=3)

X_3d = np.array(vectors_viz)

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_3d[:,0], X_3d[:,1], X_3d[:,2], s=10, alpha=0.7)

ax.set_title("3D UMAP Visualization")
ax.set_xlabel("Dim 1")
ax.set_ylabel("Dim 2")
ax.set_zlabel("Dim 3")

plt.show()

Stage-30

In [ ]:
import plotly.express as px
import pandas as pd

df_plot = pd.DataFrame({
    "x": X_3d[:,0],
    "y": X_3d[:,1],
    "z": X_3d[:,2]
})

fig = px.scatter_3d(
    df_plot,
    x="x",
    y="y",
    z="z",
    opacity=0.1,
    title="3D UMAP Visualization"
)

fig.show()

Stage-31

In [ ]:
# calculate epsilon for ITER-DBSCAN

from scipy.spatial.distance import pdist
distances = pdist(vectors_ner_power, metric='cosine')  # semua pasangan jarak

min_dist = distances.min()
median_dist = np.median(distances)
max_dist = distances.max()
    
#factor < 1.0 → cluster more noise
#factor > 1.0 → cluster less noise
factor = 1
eps_recommend = median_dist * factor
    
print(f"Min distance : {min_dist:.4f}")
print(f"Median distance : {median_dist:.4f}")
print(f"Max distance : {max_dist:.4f}")
print(f"Recommended eps (median * factor={factor}) : {eps_recommend:.4f}")

Stage-32

In [ ]:
# define MOdel ITER-DBSCAN Parameter Tuning

#%%time
model = ITER_DBSCAN(
    initial_distance=0.1,
    initial_minimum_samples=4,
    delta_distance=0.01, 
    delta_minimum_samples=1, 
    max_iteration=15,
    # features="precomputed", # Comment if fit_predict using direct dataset
    # embedding_model="IndoSBERT",  # Uncomment if fit_predict using direct dataset
    # metric="euclidean" # Uncomment if fit_predict using direct dataset
)

Stage-33: Run HDBSCAN

In [ ]:
# HDBSCAN
clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=15)
labels2 = clusterer.fit_predict(vectors_ner_power) #reduce to 120 dimension from vectors_concat and then normalized-> vectors_norm
df['cluster_ids_hdbscan'] = labels2
df.cluster_ids_hdbscan.value_counts()

Stage-34

In [ ]:
# clustering ITER-DBSCAN
#%%time
labels = model.fit_predict(vectors_ner_power) #vectors_norm, vectors_concat, vectors_reduce

Stage-35

In [ ]:
df['cluster_ids'] = labels
df.cluster_ids.value_counts()

Stage-36

In [ ]:
df.to_excel("result-clustering-hdbscan-ner-only.xlsx", index=False)

Stage-37: evaluate clustering result

In [ ]:
#evaluate clustering result

import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

X = np.stack([np.array(v, dtype=np.float32) for v in vectors_ner_power]) #vectors_norm
labels = np.array(labels2)

mask = labels != -1
X_valid = X[mask]
labels_valid = labels[mask]

print(f"Total samples     : {X.shape[0]}")
print(f"Samples w/o noise : {X_valid.shape[0]}")
print(f"Clusters considered (no -1) : {np.unique(labels_valid)}")


sil_score_global = silhouette_score(X_valid, labels_valid, metric='cosine')
print(f"\nGlobal Silhouette Score (no noise): {sil_score_global:.4f}")

# Silhouette Score Per Cluster
sil_samples = silhouette_samples(X_valid, labels_valid, metric='cosine')
cluster_summary = []

for c in np.unique(labels_valid):
    mask_c = labels_valid == c
    cluster_sil = np.mean(sil_samples[mask_c])
    cluster_size = np.sum(mask_c)
    is_negative = cluster_sil < 0
    cluster_summary.append({
        "cluster": c,
        "size": cluster_size,
        "silhouette_score": cluster_sil,
        "negative_score": is_negative
    })

df_cluster_summary = pd.DataFrame(cluster_summary)
print("\nSilhouette Score per Cluster (no noise):")
print(df_cluster_summary)

# Davies-Bouldin & Calinski-Harabasz Score
db_score = davies_bouldin_score(X_valid, labels_valid)
ch_score = calinski_harabasz_score(X_valid, labels_valid)

print(f"\nDavies-Bouldin Score      : {db_score:.4f}  (lower is better)")
print(f"Calinski-Harabasz Score   : {ch_score:.4f}  (higher is better)")

output_file = "cluster_evaluation_summary.xlsx"
df_cluster_summary.to_excel(output_file, index=False)
print(f"\nCluster summary saved to: {output_file}")

Stage-38

In [ ]:
labels_plot = np.array(labels2)

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# highlight_clusters = [0,  1,  2,  3,  4,  5,  6,  7,  8,  9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151]
highlight_clusters = [0,  1,  2,  3,  4,  5,  6,  7,  8,  9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32]

def reduce_dimensionality(vectors_source, method='umap', target_dim=3, random_state=42):
    
    X = np.stack([np.array(v, dtype=np.float32) for v in vectors_source], axis=0)
    print("Input shape:", X.shape)
    
    if method.lower() == 'umap':
        reducer = umap.UMAP(
            n_components=target_dim,
            random_state=random_state,
            n_neighbors=30,
            min_dist=0.0,
            metric='cosine'
        )
    elif method.lower() == 'tsne':
        reducer = TSNE(
            n_components=target_dim,
            method='exact',
            random_state=random_state,
            init='pca'
        )
    else:
        raise ValueError("Method must be 'umap' or 'tsne'")
    
    red_vectors = reducer.fit_transform(X)
    red_vectors_list = [np.array(v, dtype=np.float32) for v in red_vectors]
        
    return red_vectors_list


vectors_viz = reduce_dimensionality(vectors_ner_power, method='umap', target_dim=3)
X_3d = np.array(vectors_viz)

unique_labels = np.unique(labels_plot)

fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')

colors = plt.cm.tab20(np.linspace(0,1,len(unique_labels)))

for color, label in zip(colors, unique_labels):

    mask = labels_plot == label

    # Noise
    if label == -1:
        ax.scatter(
            X_3d[mask,0],
            X_3d[mask,1],
            X_3d[mask,2],
            c="black",
            s=10,
            alpha=0.3,
            label="Noise (-1)"
        )

    # Highlight cluster
    elif label in highlight_clusters:
        ax.scatter(
            X_3d[mask,0],
            X_3d[mask,1],
            X_3d[mask,2],
            c=[color],
            s=1,
            alpha=1.0,
            label=f"Highlighted Cluster {label}"
        )

    # Other clusters (dimmed)
    else:
        ax.scatter(
            X_3d[mask,0],
            X_3d[mask,1],
            X_3d[mask,2],
            c="lightgray",
            s=1,
            alpha=0.2
        )

ax.set_title("3D UMAP Visualization with Highlighted Clusters")
ax.set_xlabel("Dim 1")
ax.set_ylabel("Dim 2")
ax.set_zlabel("Dim 3")

ax.legend()
plt.show()

Stage-39

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

labels_plot = np.array(labels2)

highlight_clusters = [0,  1,  2,  3,  4,  5,  6,  7,  8,  9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32]

def reduce_dimensionality(vectors_source, method='umap', target_dim=3, random_state=42):
    
    X = np.stack([np.array(v, dtype=np.float32) for v in vectors_source], axis=0)
    print("Input shape:", X.shape)
    
    if method.lower() == 'umap':
        reducer = umap.UMAP(
            n_components=target_dim,
            random_state=random_state,
            n_neighbors=30,
            min_dist=0.0,
            metric='cosine'
        )
    elif method.lower() == 'tsne':
        reducer = TSNE(
            n_components=target_dim,
            method='exact',
            random_state=random_state,
            init='pca'
        )
    else:
        raise ValueError("Method must be 'umap' or 'tsne'")
    
    red_vectors = reducer.fit_transform(X)
    return red_vectors


# Reduce embedding → 3D
vectors_viz = reduce_dimensionality(vectors_ner_power, method='umap', target_dim=3)
X_3d = np.array(vectors_viz)

# dataframe plotly
df_plot = pd.DataFrame({
    "x": X_3d[:,0],
    "y": X_3d[:,1],
    "z": X_3d[:,2],
    "cluster": labels_plot
})

def cluster_color(row):
    if row["cluster"] == -1:
        return "Noise"
    elif row["cluster"] in highlight_clusters:
        return f"Highlighted {row['cluster']}"
    else:
        return "Other"

df_plot["category"] = df_plot.apply(cluster_color, axis=1)

# Plot interactive
fig = px.scatter_3d(
    df_plot,
    x="x",
    y="y",
    z="z",
    color="category",
    opacity=1,
    title="3D UMAP Visualization with Cluster Highlight"
)

fig.update_traces(marker=dict(size=1))

fig.show()

Stage-40

In [ ]:
df.head(5)

Stage-41

In [ ]:
# Labeling (TF-IDF)

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

TOP_TERMS = 10
TEXT_COLUMN = "entities"
CLUSTER_COLUMN = "cluster_ids_hdbscan"

valid_clusters = df[df[CLUSTER_COLUMN] != -1][CLUSTER_COLUMN].unique()
cluster_paragraphs = []

for c in valid_clusters:
    cluster_texts = df[df[CLUSTER_COLUMN] == c][TEXT_COLUMN].tolist()
    
    cluster_paragraph = " ".join(cluster_texts)
    cluster_paragraphs.append(cluster_paragraph)

# TF-IDF matrix
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(cluster_paragraphs)
feature_names = vectorizer.get_feature_names_out()

# top-N terms per cluster
cluster_labels = {}
for i, c in enumerate(valid_clusters):
    tfidf_scores = tfidf_matrix[i].toarray()[0]
    top_idx = tfidf_scores.argsort()[-TOP_TERMS:][::-1]  
    top_terms = [feature_names[j] for j in top_idx]
    cluster_labels[c] = top_terms

df_labels = pd.DataFrame({
    "cluster_id": list(cluster_labels.keys()),
    "top_terms": [", ".join(words) for words in cluster_labels.values()]
})

print("Cluster labels generated with top TF-IDF terms:")
print(df_labels.head(10))

output_file = "cluster_labels_TFIDF.xlsx"
df_labels.to_excel(output_file, index=False)
print(f"\nSaved cluster labels to: {output_file}")

Stage-42

In [ ]:
# cross cluster similarity

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

SIMILARITY_THRESHOLD = 0.7

similarity_matrix = cosine_similarity(tfidf_matrix)

df_similarity = pd.DataFrame(
    similarity_matrix,
    index=valid_clusters,
    columns=valid_clusters
)

print("\n=== Cosine Similarity Matrix Between Clusters ===")
print(df_similarity.round(3))

similar_cluster_rows = []

for i, cluster_id in enumerate(valid_clusters):

    similar_clusters = []

    for j, other_cluster in enumerate(valid_clusters):

        if i == j:
            continue

        sim_score = similarity_matrix[i, j]

        if sim_score >= SIMILARITY_THRESHOLD:
            similar_clusters.append(other_cluster)

    if similar_clusters:
        print(
            f"Cluster: {cluster_id} have similarity with cluster: "
            f"{similar_clusters} with similarity score ≥ {SIMILARITY_THRESHOLD}"
        )

    similar_cluster_rows.append({
        "cluster_id": cluster_id,
        "similar_clusters": similar_clusters
    })

df_cluster_similarity = pd.DataFrame(similar_cluster_rows)

print("\n=== Cluster Similarity Summary ===")
print(df_cluster_similarity.head())

output_file = "cluster_labels_TFIDF_2.xlsx"

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:

    df_labels.to_excel(writer, sheet_name="Cluster_Labels", index=False)
    df_similarity.to_excel(writer, sheet_name="Similarity_Matrix")
    df_cluster_similarity.to_excel(writer, sheet_name="Cluster_Similarity", index=False)

print(f"\nCluster similarity analysis saved to {output_file}")

Stage-43

In [ ]:
# cluster vizualization

import networkx as nx
import matplotlib.pyplot as plt

# ====================================
# BUAT GRAPH
# ====================================
G = nx.Graph()

# add node (cluster)
for c in valid_clusters:
    G.add_node(c)

# add edge jika similarity >= threshold
for i, cluster_i in enumerate(valid_clusters):
    for j, cluster_j in enumerate(valid_clusters):

        if i >= j:
            continue

        sim_score = similarity_matrix[i, j]

        if sim_score >= SIMILARITY_THRESHOLD:
            G.add_edge(cluster_i, cluster_j, weight=sim_score)

plt.figure(figsize=(10,8))

pos = nx.spring_layout(G, seed=42)

# node
nx.draw_networkx_nodes(
    G, pos,
    node_size=1200,
    node_color="skyblue"
)

# edges
edges = G.edges(data=True)
weights = [d['weight']*3 for (_,_,d) in edges]

nx.draw_networkx_edges(
    G, pos,
    width=weights,
    edge_color="gray"
)

# label node
nx.draw_networkx_labels(
    G, pos,
    font_size=10,
    font_weight="bold"
)

# label edge (similarity value)
edge_labels = {(u,v): f"{d['weight']:.2f}" for u,v,d in edges}

nx.draw_networkx_edge_labels(
    G, pos,
    edge_labels=edge_labels,
    font_size=8
)

plt.title("Cluster Similarity Network (TF-IDF Cosine Similarity)")
plt.axis("off")
plt.show()

Stage-44

In [ ]:
# refine cluster by cosine similarity

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# copy original label
df['cluster_id_refine'] = df['cluster_ids_hdbscan'].copy()

# create index noise
noise_idx = df[df['cluster_ids_hdbscan'] == -1].index

X = np.stack(vectors_ner_power, axis=0)

# loop each noise, assign to nearest cluster
for idx in noise_idx:
    mean_sims = []
    unique_clusters = [c for c in np.unique(df['cluster_ids_hdbscan']) if c != -1]

    for c in unique_clusters:
        cluster_points = X[df['cluster_ids_hdbscan'] == c]
        # calculate mean similarity between noise node and all nodes
        sim = np.mean(cosine_similarity([X[idx]], cluster_points))
        mean_sims.append(sim)

    # cluster with highest similarity
    closest_cluster = unique_clusters[np.argmax(mean_sims)]

    # assign to refined column
    df.loc[idx, 'cluster_id_refine'] = closest_cluster

# buat summary noise
noise_summary = df.loc[noise_idx, ['question', 'cluster_ids_hdbscan', 'cluster_id_refine']]
noise_summary['mean_similarity'] = [
    np.max([
        np.mean(cosine_similarity([X[i]], X[df['cluster_ids_hdbscan'] == c]))
        for c in unique_clusters
    ]) for i in noise_idx
]

# to Excel
output_file = "cluster_label_refinement.xlsx"
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name="All_Data", index=False)
    noise_summary.to_excel(writer, sheet_name="Noise_Summary", index=False)

print("cluster_id_refine added to df")
print("Full dataset + noise summary saved to cluster_label_refinement.xlsx")
print("\nPreview noise summary:")
df.head(5)